# Week 6 Assignment: Fine-Tuning Interface

# Fine-Tuning a Legal Research Assistant for Healthcare Access, Insurance, and Patient Billing Questions

## Project Overview

This project prepares a legal-domain dataset for a fine-tuning experiment. The focus is healthcare access, insurance, and patient billing questions. Instead of building a general legal chatbot, this project adapts a language model to respond to common questions about medical bills, insurance denials, hospital financial assistance, surprise billing, and patient rights.

The goal is not to provide legal advice. The goal is to help users understand general legal and policy-related information in plain language and suggest practical next steps when appropriate.


## Objective

The objective of this project is to demonstrate how a pre-trained language model could be adapted to a specific legal domain using fine-tuning preparation steps.

At a minimum, this project includes:

- A legal-domain dataset prepared for fine-tuning
- Sample prompt/completion pairs
- Data preprocessing steps
- Fine-tuning configuration settings
- Example legal questions and expected outputs
- A comparison between baseline ChatGPT responses and expected domain-adapted responses
- An evaluation plan for measuring model performance

A live fine-tuning experiment is not required for this assignment, so this project focuses on dataset preparation, formatting, configuration planning, and evaluation design.

*** 
## Preparation and Environment Setup


➜ **Import Libraries**

In [1]:
import pandas as pd
import json
import re
from pathlib import Path

➜ **Create Legal Text Dataset**

*Assignment Note: If your account does not have access to the fine-tuning interface, you may complete this assignment by designing and documenting a complete fine-tuning workflow rather than executing a live fine-tuning job.*

Assignment "must-have" checklist: <br>
✅ A legal-domain dataset prepared for fine-tuning <br>
✅ Sample prompt/completion pairs

In [ ]:
data = [
    {
        "question": "I received a hospital bill that is much higher than I expected. What should I do first?",
        "legal_topic": "medical billing",
        "completion": "You can start by requesting an itemized bill from the hospital and comparing it with your Explanation of Benefits from your insurance company. Look for duplicate charges, services you do not recognize, or amounts that insurance may not have processed correctly. You may also ask the hospital whether you qualify for financial assistance or a payment plan. This is general information, not legal advice. If the bill involves collections, a lawsuit, or a large disputed amount, consider contacting a patient advocate or qualified attorney."
    },
    {
        "question": "My insurance denied coverage for a procedure my doctor said was medically necessary. Can I appeal?",
        "legal_topic": "insurance denial",
        "completion": "In many cases, patients can appeal an insurance denial. A good first step is to read the denial letter carefully and identify the reason for the denial, the appeal deadline, and the documents required. You may want to ask your doctor for a letter explaining why the procedure was medically necessary. You can also request the policy language the insurer relied on. This is general information only and does not guarantee a result. If the denial affects urgent care or a major expense, consider contacting your insurance company, a patient advocate, or a qualified legal professional."
    },
    {
        "question": "Can a hospital send my medical bill to collections even if I am trying to apply for financial assistance?",
        "legal_topic": "medical debt collection",
        "completion": "Hospitals may have policies about when accounts are sent to collections, but those policies can vary. You should ask the hospital billing department whether your account can be paused while your financial assistance application is being reviewed. Keep copies of your application, letters, emails, and payment records. You may also ask for the hospital's written financial assistance and collections policies. This is general information, not legal advice. If you receive a court notice or debt collection letter, consider speaking with a qualified attorney or consumer rights organization."
    },
    {
        "question": "What is an Explanation of Benefits and is it the same as a bill?",
        "legal_topic": "insurance documents",
        "completion": "An Explanation of Benefits, often called an EOB, is a document from your insurance company that explains how a medical claim was processed. It usually shows the provider's charge, the amount allowed by insurance, what insurance paid, and what you may owe. An EOB is not the same as a bill. A bill usually comes from the hospital, clinic, or provider. You can compare the EOB and the bill to check whether the amounts match. This is general information, not legal advice."
    },
    {
        "question": "I was treated in an emergency room and later found out one of the doctors was out of network. What can I do?",
        "legal_topic": "surprise billing",
        "completion": "Emergency care can involve complicated billing, especially when a hospital or doctor is out of network. You can start by reviewing your bill and Explanation of Benefits to identify which charges were treated as out of network. You may also ask the provider and insurance company whether surprise billing protections apply. Keep copies of all bills, EOBs, and communication records. This is general information only. Because rules can vary, consider contacting your insurance company, a patient advocate, or a qualified legal professional if the amount is large or disputed."
    },
    {
        "question": "Can I ask a hospital for financial assistance if I have insurance?",
        "legal_topic": "financial assistance",
        "completion": "Yes, some hospitals allow patients with insurance to apply for financial assistance, depending on income, household size, and the remaining balance after insurance. You can ask the hospital billing office for its financial assistance policy and application. You may need to provide proof of income, insurance information, and other documents. This is general information, not legal advice. If you are unsure about eligibility, ask the hospital for the written policy and keep a copy of your application."
    },
    {
        "question": "My insurance says a service was not medically necessary. What does that mean?",
        "legal_topic": "medical necessity",
        "completion": "When insurance says a service was not medically necessary, it usually means the insurer decided the service did not meet its coverage rules for medical need. This does not always mean the service was not important. You can review the denial letter, ask for the policy criteria used in the decision, and speak with your doctor about providing medical records or a letter of medical necessity. This is general information only and not legal advice."
    },
    {
        "question": "What should I do if I cannot understand the charges on my medical bill?",
        "legal_topic": "itemized billing",
        "completion": "You can ask the provider for an itemized bill, which lists the services, dates, codes, and charges in more detail. Compare the itemized bill to your Explanation of Benefits from your insurance company. Look for duplicate charges, incorrect dates, services you do not recognize, or charges that should have been billed to insurance. This is general information, not legal advice. If the bill remains unclear, contact the billing office, your insurance company, or a patient advocate."
    }
] # This creates a list of dictionaries, where each dictionary contains a question, its associated legal topic, and a completion that provides general information related to the question. The data structure is suitable for creating a DataFrame in pandas.

df = pd.DataFrame(data) # This creates a DataFrame from the list of dictionaries, where each dictionary represents a row in the DataFrame. The keys of the dictionaries become the column names, and the values become the corresponding cell values in each row.
df # This will display the DataFrame in a tabular format, showing the questions, legal topics, and completions for each entry in the dataset.

,question,legal_topic,completion
0,I received a hospital bill that is much higher...,medical billing,You can start by requesting an itemized bill f...
1,My insurance denied coverage for a procedure m...,insurance denial,"In many cases, patients can appeal an insuranc..."
2,Can a hospital send my medical bill to collect...,medical debt collection,Hospitals may have policies about when account...
3,What is an Explanation of Benefits and is it t...,insurance documents,"An Explanation of Benefits, often called an EO..."
4,I was treated in an emergency room and later f...,surprise billing,Emergency care can involve complicated billing...
5,Can I ask a hospital for financial assistance ...,financial assistance,"Yes, some hospitals allow patients with insura..."
6,My insurance says a service was not medically ...,medical necessity,When insurance says a service was not medicall...
7,What should I do if I cannot understand the ch...,itemized billing,"You can ask the provider for an itemized bill,..."


➜ **Save the Created Dataset for Use in the Assignment**

In [3]:
Path("../data").mkdir(exist_ok=True) # This line creates a directory named "data" in the parent directory if it does not already exist. The `exist_ok=True` parameter allows the operation to succeed without raising an error if the directory already exists.

raw_path = "../data/raw_legal_healthcare_examples.csv" # This line defines the file path where the raw dataset will be saved as a CSV file. The path points to the "data" directory created earlier, and the file is named "raw_legal_healthcare_examples.csv".
df.to_csv(raw_path, index=False) #  This line saves the DataFrame `df` to a CSV file at the specified `raw_path`. The `index=False` parameter ensures that the row indices are not included in the saved CSV file, resulting in a cleaner dataset.

print(f"Raw dataset saved to: {raw_path}")

Raw dataset saved to: ../data/raw_legal_healthcare_examples.csv


➜ **Dataset Inspection Before Preprocessing**

In [4]:
df.info() # This line displays a concise summary of the DataFrame, including the number of entries, column names, non-null counts, and data types for each column. It helps to quickly understand the structure and completeness of the dataset.

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   question     8 non-null      str  
 1   legal_topic  8 non-null      str  
 2   completion   8 non-null      str  
dtypes: str(3)
memory usage: 324.0 bytes


In [6]:
df["legal_topic"].value_counts() # This line counts the occurrences of each unique value in the "legal_topic" column of the DataFrame. It provides insight into how many questions are associated with each legal topic, which can be useful for understanding the distribution of topics in the dataset. 

legal_topic
medical billing            1
insurance denial           1
medical debt collection    1
insurance documents        1
surprise billing           1
financial assistance       1
medical necessity          1
itemized billing           1
Name: count, dtype: int64

In [7]:
df.isnull().sum() # This line checks for any missing values in the DataFrame by summing the number of null entries in each column. It helps to identify if there are any incomplete records that may need to be addressed before further analysis or processing.

question       0
legal_topic    0
completion     0
dtype: int64

Dataset Overview

The dataset contains sample legal-domain questions and expected assistant responses. Each row includes a user question, a legal topic, and a completion that the fine-tuned model should learn to produce.

The dataset was checked for missing values and topic distribution before being converted into fine-tuning format.

The dataset includes:
- 3 Columns: question, legal_topic, and completion
- 8 Rows
- 0 Null values
- str data types

This dataset is a small synthetic dataset created for academic demonstration. The examples are designed to represent realistic healthcare billing and insurance questions, but they are not copied from private patient records or confidential legal documents.

***

# Dataset Preparation

Ensure the dataset is clean, well-formatted, and domain representative. <br>

Assignment "must-have" checklist: <br>
✅ Data preprocessing steps

In [10]:
def clean_text(text):
    """
    Cleans the input text by removing unwanted characters and formatting.
    
    Parameters:
    text (str): The text to be cleaned.
    
    Returns:
    str: The cleaned text.
    """
    # Remove leading/trailing whitespace
    text = text.strip()
    
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    
    # Remove any non-printable characters
    text = re.sub(r'[^\x20-\x7E]', '', text)
    
    return text

df["question_clean"] = df["question"].apply(clean_text)
df["completion_clean"] = df["completion"].apply(clean_text)

df[["question_clean", "completion_clean"]].head()

,question_clean,completion_clean
0,I received a hospital bill that is much higher...,You can start by requesting an itemized bill f...
1,My insurance denied coverage for a procedure m...,"In many cases, patients can appeal an insuranc..."
2,Can a hospital send my medical bill to collect...,Hospitals may have policies about when account...
3,What is an Explanation of Benefits and is it t...,"An Explanation of Benefits, often called an EO..."
4,I was treated in an emergency room and later f...,Emergency care can involve complicated billing...


Data Preprocessing Overview

The preprocessing step cleaned extra spaces, removed non-printable characters, and standardized the text format. Unlike some NLP preprocessing tasks, this project does not remove stopwords, punctuation, or legal and healthcare terms because the model needs complete natural-language examples for fine-tuning.

For chat-based fine-tuning, preserving sentence structure and context is important because the model learns from realistic user-assistant conversations.

***
# Fine Tuning Process

➜ **Prompt/Completion Chat Format**

In [11]:
system_message = (
    "You are a legal research assistant that explains healthcare access, "
    "insurance, and patient billing topics in plain language. "
    "You provide general legal information, not legal advice."
)

training_examples = []

for _, row in df.iterrows():
    example = {
        "messages": [
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": row["question_clean"]
            },
            {
                "role": "assistant",
                "content": row["completion_clean"]
            }
        ]
    }
    training_examples.append(example)

training_examples[0]

{'messages': [{'role': 'system',
   'content': 'You are a legal research assistant that explains healthcare access, insurance, and patient billing topics in plain language. You provide general legal information, not legal advice.'},
  {'role': 'user',
   'content': 'I received a hospital bill that is much higher than I expected. What should I do first?'},
  {'role': 'assistant',
   'content': 'You can start by requesting an itemized bill from the hospital and comparing it with your Explanation of Benefits from your insurance company. Look for duplicate charges, services you do not recognize, or amounts that insurance may not have processed correctly. You may also ask the hospital whether you qualify for financial assistance or a payment plan. This is general information, not legal advice. If the bill involves collections, a lawsuit, or a large disputed amount, consider contacting a patient advocate or qualified attorney.'}]}

➜ **Save the Fine-Tuning Dataset as JSONL**

In [12]:
jsonl_path = "../data/cleaned_prompt_completion_data.jsonl"

with open(jsonl_path, "w") as f:
    for example in training_examples:
        f.write(json.dumps(example) + "\n")

print(f"Fine-tuning dataset saved to: {jsonl_path}")
print(f"Number of training examples: {len(training_examples)}")

Fine-tuning dataset saved to: ../data/cleaned_prompt_completion_data.jsonl
Number of training examples: 8


➜ **Preview JSONL File**

In [13]:
with open(jsonl_path, "r") as f:
    for i, line in enumerate(f):
        print(line)
        if i == 2:
            break

{"messages": [{"role": "system", "content": "You are a legal research assistant that explains healthcare access, insurance, and patient billing topics in plain language. You provide general legal information, not legal advice."}, {"role": "user", "content": "I received a hospital bill that is much higher than I expected. What should I do first?"}, {"role": "assistant", "content": "You can start by requesting an itemized bill from the hospital and comparing it with your Explanation of Benefits from your insurance company. Look for duplicate charges, services you do not recognize, or amounts that insurance may not have processed correctly. You may also ask the hospital whether you qualify for financial assistance or a payment plan. This is general information, not legal advice. If the bill involves collections, a lawsuit, or a large disputed amount, consider contacting a patient advocate or qualified attorney."}]}

{"messages": [{"role": "system", "content": "You are a legal research ass

Fine-Tuning Overview

The cleaned dataset was converted into JSONL format. Each line contains a training example with three message roles:

- System: Defines the behavior of the assistant
- User: Provides the legal healthcare question
- Assistant: Provides the expected response

This structure helps the model learn how to respond consistently to healthcare access and patient billing questions.

***

# Fine-Tuning Configuration

Assignment "must-have" checklist: <br>
✅ Fine-tuning configuration settings you would use

**If I had access to OpenAI fine-tuning, I would use the following configuration:**

- training_file: cleaned_prompt_completion_data.jsonl
- model=gpt-4o-mini,
- method: supervised
- hyperparameters: {"n_epochs": 3}
- Learning rate multiplier: auto
- Batch size: Auto
- Validation split: 80/20


| Setting | Planned Value |
|---|---|
| Base Model | gpt-4o-mini |
| Method | Supervised fine-tuning |
| Training File | cleaned_prompt_completion_data.jsonl |
| File Format | JSONL chat format |
| Epochs | Auto, or 3 for a small demonstration dataset |
| Batch Size | Auto |
| Learning Rate Multiplier | Auto |
| Validation Split | 80% training / 20% validation |

***
## Example Questions & Outputs

Assignment "must-have" checklist: <br>
✅ Example legal questions and expected outputs

### Example 1

**Question:**  
I received a hospital bill that is much higher than I expected. What should I do first?

**Expected Output:**  
The assistant should recommend requesting an itemized bill, comparing it with the Explanation of Benefits, checking for duplicate or incorrect charges, asking about financial assistance, and seeking professional help if the bill is in collections or involves a lawsuit.

### Example 2

**Question:**  
My insurance denied coverage for a procedure my doctor said was medically necessary. Can I appeal?

**Expected Output:**  
The assistant should explain that many insurance denials can be appealed, recommend reviewing the denial letter, checking appeal deadlines, requesting the policy language used, and asking the doctor for a letter of medical necessity.

### Example 3

**Question:**  
Can a hospital send my bill to collections while I am applying for financial assistance?

**Expected Output:**  
The assistant should explain that hospital policies vary, recommend asking whether collections can be paused during the application review, and encourage the user to keep documentation.

***
## Baseline ChatGPT and Fine-Tuned Comparison

Assignment "must-have" checklist: <br>
✅ A comparison between baseline ChatGPT responses and responses you would expect from a domain-adapted model

### Baseline ChatGPT vs. Expected Domain-Adapted Model

| User Question | Baseline ChatGPT Response | Expected Domain-Adapted Response |
|---|---|---|
| My ER bill is much higher than expected. What should I do? | Gives general advice to contact the hospital and insurance company. | Gives healthcare-specific steps: request an itemized bill, compare with EOB, check network status, ask about financial assistance, document communications, and seek legal help if collections or lawsuits are involved. |
| My insurance denied my procedure. Can I appeal? | Explains insurance appeals generally. | Explains denial letters, appeal deadlines, medical necessity letters, internal review, external review, and plan-specific rules. |
| Can a hospital send my bill to collections? | Gives broad debt collection information. | Explains hospital billing policies, financial assistance review, written collection policies, documentation, and when to contact a consumer attorney or patient advocate. |

***
# Evaluation

Assignment "must-have" checklist: <br>
✅ An evaluation plan describing how performance would be measured

### Evaluation Plan

The model would be evaluated using both qualitative and rubric-based methods.

First, I would create a separate test set of healthcare access and patient billing questions that were not included in the training dataset. These questions would cover topics such as surprise billing, insurance denials, itemized bills, financial assistance, and medical debt collection.

Each response would be scored using the following rubric:

| Category | Description | Score Range |
|---|---|---|
| Relevance | Does the response answer the user's question? | 1-5 |
| Domain Accuracy | Does the response correctly explain healthcare billing or legal concepts? | 1-5 |
| Clarity | Is the response understandable for a non-lawyer? | 1-5 |
| Safety | Does the model avoid giving direct legal advice? | 1-5 |
| Actionability | Does the response suggest practical next steps? | 1-5 |
| Escalation | Does the response recommend professional help when appropriate? | 1-5 |

A successful domain-adapted model should score higher than the baseline model in healthcare-specific accuracy, clarity, and actionability while maintaining safe legal boundaries.

***

## Conclusion

This project demonstrates how a pre-trained language model could be adapted for a specific legal-domain use case: healthcare access, insurance, and patient billing questions. The project includes a legal-domain dataset, preprocessing steps, prompt/completion examples, JSONL formatting, fine-tuning configuration settings, baseline comparison, and an evaluation plan.

I chose this topic because healthcare billing and insurance issues are common, confusing, and stressful for many patients. Legal and policy information can be difficult to understand, especially when it involves medical bills, insurance denials, financial assistance, or collections. This project shows how fine-tuning could make a legal research assistant more focused, practical, and accessible while still maintaining safe boundaries by providing general information rather than legal advice.

Overall, the expected domain-adapted model would provide clearer, safer, and more healthcare-specific responses than a general-purpose chatbot.